In [23]:
#Load Project Environment
import Pkg
Pkg.activate(dirname(@__DIR__))
Pkg.instantiate()

  Activating project at `c:\Users\pbb62\Documents\Repositories\CHANCE_C.jl`
Precompiling project...
  ✓ CHANCE_C
  1 dependency successfully precompiled in 10 seconds. 315 already precompiled.


In [24]:
#Load Packages
using CSV, DataFrames
using DataStructures
using Agents
using Statistics,StatsBase,Distributions
using CategoricalArrays

include(joinpath(dirname(@__DIR__), "src/CHANCE_C.jl"))
using .CHANCE_C

In [25]:
###Load Input data:
##For flood history input
f_df = DataFrame(CSV.File(joinpath(dirname(@__DIR__), "data", "synth_flood_phil.csv")))

##For BG
#open bg file
phil_bg = DataFrame(CSV.File(joinpath(dirname(@__DIR__), "data/philly_bg_2019.csv")))
#groupby BG
grouped_phil_bg = groupby(phil_bg, :GEOID)

##load pop data
phil_cbsa_base_pop = DataFrame(CSV.File(joinpath(dirname(dirname(@__DIR__)), "philadelphia-data/census_data/synth_pop/pop_files/philly_cbsa_pop_0.csv")))
#drop missing values
dropmissing!(phil_cbsa_base_pop, :NP)
#drop rows with negative income
subset!(phil_cbsa_base_pop, :adj_income_2019 .=> ByRow(!<(0)))

#Subset to Phil. County (Not part of function)
phil_base_pop = subset(phil_cbsa_base_pop, :county => x -> x .== 42101)

Row,serialno,year,state,puma,rep,county,tract,bg,puma10,GEOID,RAC1P,NP,HINCP,ADJINC,adj_income_2019
,String15,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Float64?,Float64,Float64?,Float64?,Float64?
1,2015000000403,2015,42,3201,0,42101,35100,1,4203201,421010351001,1.0,3.0,100000.0,1.08047,108047.0
2,2015000000403,2015,42,3201,0,42101,35200,1,4203201,421010352001,1.0,3.0,100000.0,1.08047,108047.0
3,2015000000403,2015,42,3201,0,42101,35500,3,4203201,421010355003,1.0,3.0,100000.0,1.08047,108047.0
4,2015000000403,2015,42,3201,0,42101,35500,3,4203201,421010355003,1.0,3.0,100000.0,1.08047,108047.0
5,2015000000403,2015,42,3201,0,42101,36100,1,4203201,421010361001,1.0,3.0,100000.0,1.08047,108047.0
6,2015000000403,2015,42,3201,0,42101,36201,3,4203201,421010362013,1.0,3.0,100000.0,1.08047,108047.0
7,2015000000403,2015,42,3201,0,42101,36202,3,4203201,421010362023,1.0,3.0,100000.0,1.08047,108047.0
8,2015000000403,2015,42,3201,0,42101,36202,3,4203201,421010362023,1.0,3.0,100000.0,1.08047,108047.0
9,2015000000403,2015,42,3201,0,42101,36202,3,4203201,421010362023,1.0,3.0,100000.0,1.08047,108047.0


In [ ]:
names(grouped_phil_bg)

In [26]:
#Define input Parameters
no_of_years = 38
start_year = 1981
no_hhs_per_agent=10
growth_rate = 0.01
grouped = true
group_col = "adj_income_2019"
cutoff_dict = OrderedDict("low"=> [-60000.00,25000.00], "medium"=>[25000.00,75000.00], "high"=>[75000.00, 1e7])
bg_cat = Dict(:col =>"income_cat", :group => ["low", "medium", "high"])
house_budget_mode = "perc"
house_choice_mode = "flood_mem_utility"
risk_averse = 0.3
flood_mem = 10
seed = 1500

1500

In [27]:
### Calculate Flood matrix and Dict for ABM input
f_dict, f_matrix = CHANCE_C.flood_history(f_df; no_of_years = no_of_years, start_year = start_year)

([0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0;;;], Dict(5 => (1, 5), 16 => (1, 16), 20 => (1, 20), 35 => (1, 35), 12 => (1, 12), 24 => (1, 24), 28 => (1, 28), 8 => (1, 8), 17 => (1, 17), 30 => (1, 30)…))

In [35]:
### Initialize ABM
phil_abm = CHANCE_C.Simulator(phil_bg, phil_base_pop, f_dict, f_matrix, CHANCE_C.model_step!; no_of_years = no_of_years, no_hhs_per_agent = no_hhs_per_agent,
house_budget_mode = house_budget_mode, house_choice_mode = house_choice_mode, grouped = grouped, group_col = group_col, cutoff_dict = cutoff_dict, bg_cat = bg_cat,
risk_averse = risk_averse, flood_mem = flood_mem, seed = seed)

StandardABM with 93010 agents of type Union{Main.CHANCE_C.BlockGroup, Main.CHANCE_C.HHAgent, Main.CHANCE_C.Queue}
 agents container: Dict
 space: GridSpace with size (37, 37), metric=chebyshev, periodic=true
 scheduler: Agents.Schedulers.ByType
 properties: df, total_population, flood_hazard, agent_creation, relo_sampler, agent_relocate, build_develop, house_price, hh_utilities_df, no_of_years, flood_matrix, flood_dict, tick

In [ ]:
length([a for a in allagents(phil_abm) if a isa HHAgent])

In [ ]:
function AgentMigration(model::ABM; growth_rate = 0.01)
    if growth_rate == 0.0 #In-migration not considered
        return #do nothing
    else
        migrant_ids = [a.id for a in agents_in_position(model[-1].pos, model) if a isa CHANCE_C.HHAgent]
        no_new_agents = floor(Int64, (length([a for a in allagents(model) if a isa CHANCE_C.HHAgent]) - length(migrant_ids)) * growth_rate)
        #Sample from migrant agent pool 
        incoming_ids = sample(abmrng(model), migrant_ids, no_new_agents)
        #move agents to relocation queue
        for id in incoming_ids
            move_agent!(model[id], model[0].pos, model)
        end
    end
end

AgentMigration (generic function with 1 method)

In [41]:
migrant_ids = [a.id for a in agents_in_position(phil_abm[-1].pos, phil_abm) if a isa CHANCE_C.HHAgent]
no_new_agents = floor(Int64, (length([a for a in allagents(phil_abm) if a isa CHANCE_C.HHAgent]) - length(migrant_ids)) * growth_rate)

627

In [43]:
phil_abm.tick += 1
AgentMigration(phil_abm; growth_rate = 0.01)

627-element Vector{Int64}:
 83608
 85533
 70820
 71459
 64759
 69336
 74715
 74952
 64984
 70850
     ⋮
 87792
 85494
 76329
 70589
 92909
 85456
 85135
 65869
 67454

In [46]:
collect(agents_in_position(phil_abm[0].pos, phil_abm))

625-element Vector{AbstractAgent}:
 Main.CHANCE_C.Queue(0, (26, 35), :relocating)
 Main.CHANCE_C.HHAgent(83608, (26, 35), -1, 10, "medium", 2.0, 1, 37132.272000000004, Dict(-1 => 0.0), "perc", 0, 0.95, true, 49385.92176000001, 0.33)
 Main.CHANCE_C.HHAgent(85533, (26, 35), -1, 10, "high", 1.0, 4, 608814.8620500002, Dict(-1 => 0.0), "perc", 0, 0.95, true, 809723.7665265003, 0.33)
 Main.CHANCE_C.HHAgent(70820, (26, 35), -1, 10, "medium", 2.0, 1, 64406.93999999999, Dict(-1 => 0.0), "perc", 0, 0.95, true, 85661.23019999999, 0.33)
 Main.CHANCE_C.HHAgent(71459, (26, 35), -1, 10, "medium", 1.0, 8, 31559.4006, Dict(-1 => 0.0), "perc", 0, 0.95, true, 41974.002798, 0.33)
 Main.CHANCE_C.HHAgent(64759, (26, 35), -1, 10, "low", 1.0, 1, 8870.487200000001, Dict(-1 => 0.0), "perc", 0, 0.95, true, 11797.747976000002, 0.33)
 Main.CHANCE_C.HHAgent(69336, (26, 35), -1, 10, "medium", 2.0, 2, 40391.4098, Dict(-1 => 0.0), "perc", 0, 0.95, true, 53720.575034, 0.33)
 Main.CHANCE_C.HHAgent(74715, (26, 35), -1, 1

In [47]:
function AgentLocation(agent::CHANCE_C.Queue, model::ABM; levee = false, f_e = 0.0, bg_sample_size = 10, house_choice_mode = "simple_anova_utility",
    budget_reduction_perc = 0.10, penalty = 50, migrate_prob = 0.05)

    loc_df = copy(model.df)
    # Create a GEOID-to-BlockGroup lookup
    geoid_to_bg = Dict{Int64, Int64}()
    for bg in allagents(model)
        if bg isa CHANCE_C.BlockGroup
            geoid_to_bg[bg.GEOID] = bg.id
        end
    end

    # Use view or filter instead of multiple list comprehensions
    moving_agents = sort!([a for a in agents_in_position(agent, model) if a isa CHANCE_C.HHAgent], by=a -> a.income, rev=true)

    current_index = 1
    #Preallocate some vectors to reduce memory allocations
    hh_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
    bg_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
    bg_GEOID = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
    bg_cat = Vector{String}(undef, bg_sample_size* length(moving_agents))
    bg_utilities = Vector{Float64}(undef, bg_sample_size * length(moving_agents))

    for hh_agent in moving_agents
        # Consolidate budget selection logic
        bg_budget = if house_choice_mode == "simple_avoidance_utility"
            hh_agent.avoidance ? 
                subset(loc_df, :perc_fld_area => n -> n .<= 0.10) :
                subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true)
        elseif house_choice_mode == "budget_reduction"
            new_house_budget = hh_agent.house_budget * (1 - budget_reduction_perc)
            hh_budget = ifelse.(loc_df.perc_fld_area .>= 0.10, new_house_budget, hh_agent.house_budget)
            subset(loc_df, :market_value => n -> n .<= hh_budget, skipmissing=true, view = true)
        else
            subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true, view = true)
        end


        # Use a more efficient sampling approach
        try
            # Precompute weights to avoid repeated calculations
            weights = ProbabilityWeights(bg_budget.available_units ./ sum(bg_budget.available_units))
                
            # Check for available locations more efficiently
            valid_locations = findall(weights .> 0)
            if isempty(valid_locations)
                throw(ErrorException("No affordable locations with available units"))
            end

            #Sample from affordable locations based on weights
            sample_size = min(length(valid_locations), bg_sample_size)
            sampled_indices = sample(abmrng(model), valid_locations, sample_size, replace=false)
            
            #Grab utilities from sampled locations
            loc_utilities = [model[geoid_to_bg[row.GEOID]].current_utility[row.income_cat] for row in eachrow(bg_budget[sampled_indices, [:GEOID, :income_cat]])]
            # Find indices of block groups with better utilities than current agent location
            current_utility = first(values(hh_agent.utility))
            opt_locs = findall(>(current_utility), loc_utilities)

            # Check if any moves are possible
            if isempty(opt_locs)
                throw(ErrorException("No better locations found"))
            end
            best_indices = sampled_indices[opt_locs]
            
            #Append future block group properties to vectors
            ind_length = length(best_indices)

            copyto!(hh_ids, current_index, fill(hh_agent.id, ind_length), 1, ind_length)
            copyto!(bg_ids, current_index, getindex.(Ref(geoid_to_bg), bg_budget[best_indices,:GEOID]), 1, ind_length)
            copyto!(bg_GEOID, current_index, bg_budget[best_indices, :GEOID], 1, ind_length)
            copyto!(bg_cat, current_index, bg_budget[best_indices, :income_cat], 1, ind_length)
            copyto!(bg_utilities, current_index, loc_utilities[opt_locs], 1, ind_length)

            current_index += ind_length
            
        catch
            # Migration logic remains similar
            last_bg = model[first(keys(hh_agent.utility))]
            if last_bg == -1
                remove_agent!(hh_agent, model)
                continue
            end
            
            if rand(abmrng(model), Binomial(1, migrate_prob)) == 1
                move_agent!(hh_agent, last_bg.pos, model)
                last_bg.occupied_units[hh_agent.group] += 1
                last_bg.available_units[hh_agent.group] -= 1
                last_bg.population += getproperty(hh_agent, :no_hhs_per_agent) * getproperty(hh_agent, :hh_size)
            else
                remove_agent!(hh_agent, model)
            end
        end
    end
    
    ##Create df from vectors, append to model properties df
    #Remove extra undef values by using current index
    bg_sample = DataFrame(hh_id = hh_ids[1:current_index-1], bg_id = bg_ids[1:current_index-1], 
    GEOID = bg_GEOID[1:current_index-1], cat = bg_cat[1:current_index-1], bg_utility = bg_utilities[1:current_index-1])
    
    append!(model.hh_utilities_df, bg_sample)
end

AgentLocation (generic function with 1 method)

In [48]:
phil_abm.hh_utilities_df

Row,hh_id,bg_id,GEOID,cat,bg_utility
,Int64,Int64,Int64,String,Float64


In [ ]:
CHANCE_C.AgentLocation(phil_abm[0], phil_abm)

MethodError: MethodError: no method matching AgentLocation(::StandardABM{GridSpace{2, true}, Union{Main.CHANCE_C.BlockGroup, Main.CHANCE_C.HHAgent, Main.CHANCE_C.Queue}, Dict{Int64, Union{Main.CHANCE_C.BlockGroup, Main.CHANCE_C.HHAgent, Main.CHANCE_C.Queue}}, Tuple{DataType, DataType, DataType}, typeof(dummystep), typeof(Main.CHANCE_C.model_step!), Agents.Schedulers.ByType, Main.CHANCE_C.Properties{DataFrame, Int64, Dict{Symbol, Integer}, Dict{Symbol, Float64}, Dict{Symbol, Any}, Dict{Symbol, Any}, Dict{Symbol, Any}, Dict{Symbol, Any}, DataFrame, Int64, Array{Float64, 3}, Dict{Int64, Tuple{Int64, Int64}}, Int64}, Random.MersenneTwister})

Closest candidates are:
  AgentLocation(!Matched::Main.CHANCE_C.Queue, !Matched::AgentBasedModel; levee, f_e, bg_sample_size, house_choice_mode, budget_reduction_perc, penalty, migrate_prob)
   @ Main.CHANCE_C c:\Users\pbb62\Documents\Repositories\CHANCE_C.jl\src\agent_functions\agent_relocation.jl:73


In [19]:
##Extract pop characteristics from pop_df
pop_df = copy(phil_cbsa_base_pop)
group_col = "adj_income_2019"
cutoffs = OrderedDict("low"=> [0,25000.00], "medium"=>[25000.00,75000.00], "high"=>[75000.00, 1e7])
no_hhs_per_agent = 10
#Subset to only occupied households
pop_hh_df = subset(pop_df, :NP => x -> x .> 0.0)

#Create group labels by group col
pop_hh_df[:, :category] = cut(pop_hh_df[:, group_col], unique(reduce(vcat, collect(values(cutoffs)))), labels = collect(keys(cutoffs)))
#groupby category column 
pop_cat_df = groupby(pop_hh_df, :category)

#Create empty DataFrame
agent_df = DataFrame(nrow = Int64[], cat = String[], race = Float64[], avg_hh_size = Float64[], avg_income = Float64[])
for (i,sub_df) in enumerate(pop_cat_df)
    sort!(sub_df, :adj_income_2019)
    sub_df[:,:group] = map(x->div(x,no_hhs_per_agent), 1:nrow(sub_df))
    hh_bins = combine(groupby(sub_df, :group), nrow, :category => maximum => :cat, :RAC1P => (r -> mode(r)) => :race,  [:NP, :adj_income_2019] .=> mean .=> [:avg_hh_size, :avg_income])
    inc_w = ProbabilityWeights(hh_bins.avg_income ./ sum(hh_bins.avg_income)) #Calculate weights based on avg income
    append!(agent_df, hh_bins[sample(abmrng(phil_abm), 1:nrow(hh_bins), inc_w, migrant_cat_pop[i]; replace = true),2:end]) #Sample rows based on migrant category count
end

In [22]:
agent_df

Row,nrow,cat,race,avg_hh_size,avg_income
,Int64,String,Float64,Float64,Float64
1,10,low,6.0,2.2,14239.5
2,10,low,1.0,2.0,22542.4
3,10,low,1.0,1.0,23233.3
4,10,low,2.0,1.0,11056.5
5,10,low,1.0,4.0,17282.5
6,10,low,2.0,4.0,18788.7
7,10,low,2.0,2.0,14169.5
8,10,low,1.0,2.0,13182.6
9,10,low,8.0,5.2,23723.4


In [ ]:
t_g = groupby(agent_df, :cat)[1]
inc_w = ProbabilityWeights(t_g.avg_income ./ sum(t_g.avg_income))
t_g[sample(abmrng(phil_abm), 1:nrow(t_g), inc_w, migrant_cat_pop[1]; replace = true), :]

In [ ]:
### Test model functions
test_bg = phil_abm[10]
println("Occupied: ",test_bg.occupied_units)
println("Available: ",test_bg.available_units)
println("Population: ",test_bg.population)

In [ ]:
length([a for a in agents_in_position(test_bg, phil_abm) if a isa HHAgent && a.group == "medium"])

In [ ]:
CHANCE_C.agent_prob!(test_bg, phil_abm)

In [ ]:
println("Occupied: ",test_bg.occupied_units)
println("Available: ",test_bg.available_units)
println("Population: ",test_bg.population)

In [ ]:
collect(agents_in_position(phil_abm[0].pos, phil_abm))

In [ ]:
function agent_locate(agent::CHANCE_C.Queue, model::ABM; levee = false, f_e = 0.0, bg_sample_size = 10, house_choice_mode = "simple_anova_utility",
    budget_reduction_perc = 0.10, penalty = 50, migrate_prob = 0.05)
    
    loc_df = copy(model.df)
    # Preallocate the DataFrame with a reasonable initial capacity
    #bg_sample = DataFrame(hh_id = Int64[], bg_id = Int64[], GEOID = Int64[], cat = String[], bg_utility = Float64[])

    # Use view or filter instead of multiple list comprehensions
    moving_agents = sort!([a for a in agents_in_position(agent, model) if a isa HHAgent], by=a -> a.income, rev=true)

    current_index = 1
    #Preallocate some vectors to reduce memory allocations
    hh_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
    bg_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
    bg_GEOID = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
    bg_cat = Vector{String}(undef, bg_sample_size* length(moving_agents))
    bg_utilities = Vector{Float64}(undef, bg_sample_size * length(moving_agents))

    for hh_agent in moving_agents
        # Consolidate budget selection logic
        bg_budget = if house_choice_mode == "simple_avoidance_utility"
            hh_agent.avoidance ? 
                subset(loc_df, :perc_fld_area => n -> n .<= 0.10) :
                subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true)
        elseif house_choice_mode == "budget_reduction"
            new_house_budget = hh_agent.house_budget * (1 - budget_reduction_perc)
            hh_budget = ifelse.(loc_df.perc_fld_area .>= 0.10, new_house_budget, hh_agent.house_budget)
            subset(loc_df, :market_value => n -> n .<= hh_budget, skipmissing=true, view = true)
        else
            subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true, view = true)
        end

        # Use a more efficient sampling approach
        try
            
            # Precompute weights to avoid repeated calculations
            weights = ProbabilityWeights(bg_budget.available_units ./ sum(bg_budget.available_units))
            
            # Check for available locations more efficiently
            valid_locations = findall(weights .> 0)
            if isempty(valid_locations)
                throw(ErrorException("No affordable locations with available units"))
            end

            #Sample from affordable locations based on weights
            sample_size = min(length(valid_locations), bg_sample_size)
            sampled_indices = sample(abmrng(model), valid_locations, sample_size, replace=false)
    
            #Grab utilities from sampled locations
            bg_sel = Iterators.filter(bg -> bg isa BlockGroup && bg.GEOID in bg_budget[sampled_indices, :GEOID], allagents(model)).id
            loc_utilities = getindex.(getproperty.(getindex.(Ref(model), bg_sel), :current_utility), bg_budget[sampled_indices, :income_cat])
            # Find indices of block groups with better utilities than current agent location
            current_utility = first(values(hh_agent.utility))
            opt_locs = findall(>(current_utility), loc_utilities)

            # Check if any moves are possible
            if isempty(opt_locs)
                throw(ErrorException("No better locations found"))
            end
            best_indices = sampled_indices[opt_locs]

            #Append future block group properties to vectors
            ind_length = length(best_indices)

            copyto!(hh_ids, current_index, fill(hh_agent.id, ind_length), 1, ind_length)
            copyto!(bg_ids, current_index, collect(bg_sel)[opt_locs], 1, ind_length)
            copyto!(bg_GEOID, current_index, bg_budget[best_indices, :GEOID], 1, ind_length)
            copyto!(bg_cat, current_index, bg_budget[best_indices, :income_cat], 1, ind_length)
            copyto!(bg_utilities, current_index, loc_utilities[opt_locs], 1, ind_length)

            current_index += ind_length

        catch
            # Migration logic remains similar
            if rand(abmrng(model), Binomial(1, migrate_prob)) == 1
                last_bg = model[first(keys(hh_agent.utility))]
                move_agent!(hh_agent, last_bg.pos, model)
                last_bg.occupied_units[hh_agent.group] += 1
                last_bg.available_units[hh_agent.group] -= 1
                last_bg.population += getproperty(hh_agent, :no_hhs_per_agent) * getproperty(hh_agent, :hh_size)
            else
                remove_agent!(hh_agent, model)
            end
        end
    end
    
    ##Create df from vectors, append to model properties df
    #Remove extra undef values by using current index
    bg_sample = DataFrame(hh_id = hh_ids[1:current_index-1], bg_id = bg_ids[1:current_index-1], 
    GEOID = bg_GEOID[1:current_index-1], cat = bg_cat[1:current_index-1], bg_utility = bg_utilities[1:current_index-1])
    
    append!(model.hh_utilities_df, bg_sample)
end

In [ ]:
agent_locate(phil_abm[0], phil_abm)


In [ ]:
phil_abm.hh_utilities_df

In [ ]:
#Breakdown agent relocation function:
loc_df = copy(phil_abm.df)
#Create a GEOID-to-BlockGroup lookup
geoid_to_bg = Dict{Int64, Int64}()
for bg in allagents(phil_abm)
    if bg isa BlockGroup
        geoid_to_bg[bg.GEOID] = bg.id
    end
end
house_choice_mode = "simple_anova_utility"
bg_sample_size = 10
# Use view or filter instead of multiple list comprehensions
moving_agents = sort!([a for a in agents_in_position(phil_abm[0], phil_abm) if a isa CHANCE_C.HHAgent], by=a -> a.income, rev=true)
agent_sel = moving_agents[1]
current_index = 1
# Preallocate some vectors to reduce memory allocations
bg_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
bg_GEOID = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
bg_cat = Vector{String}(undef, bg_sample_size* length(moving_agents))
bg_utilities = Vector{Float64}(undef, bg_sample_size * length(moving_agents))

# Consolidate budget selection logic
bg_budget = if house_choice_mode == "simple_avoidance_utility"
    agent_sel.avoidance ? 
        subset(loc_df, :perc_fld_area => n -> n .<= 0.10) :
        subset(loc_df, :market_value => n -> n .<= agent_sel.house_budget, skipmissing=true)
elseif house_choice_mode == "budget_reduction"
    new_house_budget = agent_sel.house_budget * (1 - budget_reduction_perc)
    hh_budget = ifelse.(loc_df.perc_fld_area .>= 0.10, new_house_budget, agent_sel.house_budget)
    subset(loc_df, :market_value => n -> n .<= hh_budget, skipmissing=true, view = true)
else
    subset(loc_df, :market_value => n -> n .<= agent_sel.house_budget, skipmissing=true, view = true)
end




In [ ]:
# Use a more efficient sampling approach
#try
    # Precompute weights to avoid repeated calculations
weights = ProbabilityWeights(bg_budget.available_units ./ sum(bg_budget.available_units))
            
    # Check for available locations more efficiently
valid_locations = findall(weights .> 0)
if isempty(valid_locations)
    throw(ErrorException("No affordable locations with available units"))
end
    # Efficient sampling
sample_size = min(length(valid_locations), bg_sample_size)
sampled_indices = sample(abmrng(phil_abm), valid_locations, sample_size, replace=false)
    #bg_options = bg_budget[sampled_indices, :]
    
    #Grab utilities from sampled locations
#bg_sel = map(bg -> bg.id, Iterators.filter(bg -> bg isa BlockGroup && bg.GEOID in bg_budget[sampled_indices, :GEOID], allagents(phil_abm)))
loc_utilities = [phil_abm[geoid_to_bg[row.GEOID]].current_utility[row.income_cat] for row in eachrow(bg_budget[sampled_indices, [:GEOID, :income_cat]])]
# Find indices of block groups with better utilities than current agent location
current_utility = first(values(agent_sel.utility))
opt_locs = findall(>(current_utility), loc_utilities)
# Check if any moves are possible
if isempty(opt_locs)
    throw(ErrorException("No better locations found"))
end
best_indices = sampled_indices[opt_locs]
#catch
#    println("didnt work!")
#end

In [ ]:
println(sampled_indices)
println(opt_locs)
println(best_indices)


In [ ]:
getindex.(Ref(geoid_to_bg), bg_budget[best_indices,:GEOID])

In [ ]:
ind_length = length(best_indices)   
#bg_ids = getproperty.(bg_sel[opt_locs], :id)
#bg_GEOID = bg_budget[sampled_indices, :GEOID]
#bg_cat = bg_budget[sampled_indices, :income_cat]
#bg_utilities = loc_utilities[opt_locs]

copyto!(bg_ids, current_index, getindex.(Ref(geoid_to_bg), bg_budget[best_indices,:GEOID]), 1, ind_length)
copyto!(bg_GEOID, current_index, bg_budget[best_indices, :GEOID], 1, ind_length)
copyto!(bg_cat, current_index, bg_budget[best_indices, :income_cat], 1, ind_length)
copyto!(bg_utilities, current_index, loc_utilities[opt_locs], 1, ind_length)
current_index += ind_length


# Append to bg_sample
#append!(bg_sample, move_df)

In [ ]:
fill(agent_sel.id, ind_length)

In [ ]:
agent_locate(phil_abm[0], phil_abm)

In [ ]:
agent_relocate(phil_abm[0], phil_abm)

In [ ]:
sort!(filter(a -> a isa HHAgent, agents_in_position(phil_abm[0], phil_abm)), by=a -> a.income, rev=true)

In [ ]:
step!(phil_abm)

In [ ]:
phil_abm.tick